# (02) Stroop PVAE task

device = ```cuda:3```

**Motivation**: <br>

In [1]:
# HIDE CODE


import os, pathlib
from IPython.display import display

# tmp & extras dir
_jb_dir = pathlib.Path(os.environ['HOME'])
_jb_dir /= 'Dropbox/git/jb-progress-2026'
extras_dir = _jb_dir / '_extras'
fig_base_dir = _jb_dir / 'figs'
tmp_dir = _jb_dir / 'tmp'

from mcfe.utils.plotting import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

In [2]:
device_idx = 3
device = f'cuda:{device_idx}'

print(f"device: {device}  ———  host: {os.uname().nodename}")

device: cuda:3  ———  host: mach

In [3]:
sys.path.insert(0, pjoin(os.environ['HOME'], 'Dropbox/git', '_StroopPVAE'))

from stroop_model import *
from stroop_train import *

In [5]:
cfg = StroopConfig(
    model_type="pois",
    encoder_type="mlp",
    mlp_hidden=256,
    latent_channels=32,
    beta_kl=0.2,
    congruent_prob=1.0,
    phases=[(5, 1.0), (5, 0.0)],   # 20ep digit, 80ep color-only
    kl_anneal_portion=0.0,
    temp_anneal_portion=0.2,
    use_scheduler=False,
    lr=0.001,
    seed=0,
    no_wandb=True,
)

In [6]:
summary, _, lit = run_single(cfg)

Seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA RTX 6000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_pre

ColoredMNISTStroop: 24754 samples, 4 classes, P(digit)=1.0, P(cong)=1.0
ColoredMNISTStroop: 4157 samples, 4 classes, P(digit)=1.0, P(cong)=1.0
StroopEvalDataset: 800 samples, task_cue=1, condition=congruent
StroopEvalDataset: 2400 samples, task_cue=1, condition=incongruent


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ StroopVAE        │  877 K │ train │     0 │
│ 1 │ ce_loss │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 877 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 877 K                                                                                                
Total estimated model params size (MB): 3                                                                          
Modules in train mode: 10                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

→ Phase 2: epoch 5, P(digit)=0.0
`Trainer.fit` stopped: `max_epochs=10` reached.



pois  β=0.2  seed=0
  FR cong:   1.48 ± 0.12
  FR incong: 1.48 ± 0.15
  Δ FR:      0.00
  Acc cong:  0.9900
  Acc incong:0.0025



In [6]:
print({k: v for k, v in summary.items() if 'per_neuron_fr' not in k})

{
    'fr_congruent_mean': 32.3029899597168,
    'fr_congruent_std': 0.04642338678240776,
    'fr_incongruent_mean': 32.31219482421875,
    'fr_incongruent_std': 0.055355627089738846,
    'congruency_effect': 0.009204864501953125,
    'acc_congruent': 0.9975000023841858,
    'acc_incongruent': 0.9399999976158142,
    'acc_effect': 0.05750000476837158,
    'kl_congruent_mean': 0.10561489313840866,
    'kl_incongruent_mean': 0.09611351788043976
}

In [17]:
cfg = StroopConfig(
    model_type="pois",
    beta_kl=2.0,
    beta_recon=1.0,
    n_classes=4,
    use_deep_encoder=True,
    latent_channels=32,
    latent_pixels=7,
    epochs=10,
    seed=0,
    no_wandb=True,
)

summary, _, lit = run_single(cfg)

Seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


ColoredMNISTStroop: 24754 samples, 4 classes, P(digit)=0.9
ColoredMNISTStroop: 4157 samples, 4 classes, P(digit)=0.9
StroopEvalDataset: 800 samples, task_cue=1, condition=congruent
StroopEvalDataset: 2400 samples, task_cue=1, condition=incongruent


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ StroopVAE        │  4.0 M │ train │     0 │
│ 1 │ ce_loss │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 4.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.0 M                                                                                                
Total estimated model params size (MB): 15                                                                         
Modules in train mode: 42                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=10` reached.

Detected KeyboardInterrupt, attempting graceful shutdown ...


RuntimeError: Please call `iter(combined_loader)` first.

In [ ]:

	
	

# %% [markdown]
# ## Full sweep (uncomment to run)
#
# ```python
# all_pois, avg_pois = run_beta_sweep(
#     model_type="pois",
#     betas=[0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
#     seeds=[0, 1, 2, 3, 4],
#     epochs=80, no_wandb=False)
#
# all_gaus, avg_gaus = run_beta_sweep(
#     model_type="gaus",
#     betas=[0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
#     seeds=[0, 1, 2, 3, 4],
#     latent_act="relu",
#     epochs=80, no_wandb=False)
#
# fig = plot_comparison(avg_pois, avg_gaus)
# fig.savefig("stroop_comparison.pdf", bbox_inches="tight", dpi=300)
# plt.show()
# ```